# Spin-1 XY evidence for Sec. 6

The notebook follows the manuscript evidence order: exact local channels (C1--C2), a generic finite-energy same-Hamiltonian microcanonical test (T1), the matched $D=0$ / $\beta=0$ extension (T2), a finite-temperature canonical check (T3), and preserving deformations with recentered thermal ensembles (T4).

## Imports and run controls

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as la

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.caging import (
    LocalWitnessTemplate,
    adjacent_gap_ratio_report,
    basis_permutation_from_variable_permutation,
    cage_compatibility_hierarchy_from_hamiltonians,
    cage_jacobian_conditioning_from_hamiltonian,
    cyclic_symmetry_sector_basis,
    diagnose_cage_stability,
    diagnose_eigenpair,
    diagnose_local_channel_spectrum,
    directed_transition_witness_template,
    eigenstate_expectations,
    gaussian_spectral_filter,
    evaluate_local_witness_on_diagonal_ensemble,
    evaluate_local_witness_on_states,
    hermitianize_local_witness_template,
    linearized_cage_obstruction,
    permutation_matrix,
    project_operator_to_sector,
    project_state_to_sector,
    refine_sector_by_involution,
    select_microcanonical_window_by_count,
    select_microcanonical_window_by_width,
    spectral_observable_moments,
    thermal_activity_margin_from_samples,
    thermodynamic_energy_window_plan,
)
from qlinks.models import (
    SpinOneXYChainModel,
    spin_one_xy_periodic_range_couplings,
    spin_one_xy_phase_compatibility,
    spin_one_xy_scar_tower_states,
    spin_one_xy_tower_thermal_activities,
)
from helpers import (
    PRX_FOUR_PANEL_FIGSIZE,
    PRX_SINGLE_PANEL_FIGSIZE,
    PRX_TWO_PANEL_FIGSIZE,
    PRX_WIDE_FIGSIZE,
    add_panel_label,
    canonical_beta_match,
    degeneracy_resolved_concentration,
    projector_deleted_observable_moments,
    save_prx_figure,
    set_revtex_matplotlib_style,
    write_figure_manifest,
)

DATA_DIR = ROOT / "experimental" / "data" / "spin1_xy_draft_evidence"
FIGURE_DIR = DATA_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TOL = 1.0e-10
RUN_PROFILE = "smoke"  # "smoke", "known", or "production"
USE_TEX = False  # set True only for the final lightweight plotting pass
PROFILE_SIZES = {
    "smoke": (8,),
    "known": (8, 10, 12),
    # Dense ED at L=14 can require O(100 GB) once projected operators,
    # eigensolver work arrays, and finite-D comparisons are all present.
    # Keep the unattended production profile safe by default; request L=14
    # explicitly through the batch job with --sizes 8,10,12,14.
    "production": (8, 10, 12),
}
if RUN_PROFILE not in PROFILE_SIZES:
    raise ValueError(f"unknown RUN_PROFILE={RUN_PROFILE!r}")
MICROCANONICAL_SIZES = PROFILE_SIZES[RUN_PROFILE]
DEFORMATION_SIZES = MICROCANONICAL_SIZES
COUNTING_LENGTHS = tuple(range(4, 32, 2))
# Backward-compatible alias used by older analysis cells.
SIZES = MICROCANONICAL_SIZES

# Unfinished-work controls from EVIDENCE_SUMMARY.md.  All are inexpensive in
# the smoke profile; the size list controls the production cost.
RUN_PROTOCOL_M = True
RUN_BACKGROUND_CONCENTRATION = True
RUN_COMPLEX_HERMITIAN_PATH = True
RUN_JOINT_CONTINUATION_CROSSCHECK = True

TOTAL_SZ = -2
J_DRAFT = 1.0
J1_MATRIX = 2.0 * J_DRAFT  # qlinks matrix element; draft bond action is 2J
J3_OVER_J = 0.10
J3_MATRIX = 2.0 * J3_OVER_J * J_DRAFT
D_THERMAL = 0.63
WINDOW_PREFACTORS = (0.75, 1.0, 1.25)
PRIMARY_WINDOW_PREFACTOR = 1.0
SMOOTH_SIGMA_PREFACTOR = 1.0

print("data directory:", DATA_DIR)
print("run profile:", RUN_PROFILE, "sizes:", SIZES, "fixed total Sz:", TOTAL_SZ)
print("draft J:", J_DRAFT, "qlinks nearest-neighbor matrix element:", J1_MATRIX)

set_revtex_matplotlib_style(base_font_size=8, prefer_tex=USE_TEX)


FIGURE_FORMATS = ("pdf", "svg")

def save_spin_figure(fig, stem: str, *, aliases=()):
    save_prx_figure(fig, stem, directory=FIGURE_DIR, formats=FIGURE_FORMATS)
    for alias in aliases:
        if alias != stem:
            save_prx_figure(fig, alias, directory=FIGURE_DIR, formats=FIGURE_FORMATS)



## Evidence map and run products

The witness order is fixed throughout as $A_R$, $Z_R=A_R+A_R^\dagger$, and $Y_R$.

- **C1--C2:** explicit bounded operators and exact tower darkness.
- **T1 / C3a:** finite-$D$ energy-matched microcanonical sequence and background concentration.
- **T2 / C3b:** $D=0$ microcanonical--$\beta=0$ overlap and exact fixed-$M$ continuation.
- **T3 / C3c:** matched finite inverse temperature.
- **C7:** basis-independent projector-deletion control.
- **T4 / C4--C6:** finite preserving paths, joint local-channel continuation, and thermal margins.

`RUN_PROFILE="known"` reproduces the cached $L=8,10,12$ evidence. Larger ED sizes are optional job parameters.

## Model, local witnesses, and symmetry-sector helpers

In [ ]:
def make_spin1_witnesses(*, xy_matrix_element: float = J1_MATRIX):
    # Y_r=(Sz_r)^2-1 is represented on its only nonzero local pattern |0>.
    y_template = LocalWitnessTemplate(
        pattern_key=(),
        local_patterns=((0,),),
        local_operator=np.asarray([[-1.0]], dtype=np.complex128),
        metadata={"name": "Y_r", "support_sites": 1, "channel_type": "diagonal"},
    )

    # A= c |00>(<+ -|+<- +|), with c=2J in the manuscript convention.
    a_template = directed_transition_witness_template(
        target_pattern=(0, 0),
        source_patterns=((1, -1), (-1, 1)),
        amplitudes=(xy_matrix_element, xy_matrix_element),
        metadata={"name": "Ared_r_r+1", "support_sites": 2},
    )
    z_template = hermitianize_local_witness_template(
        a_template,
        metadata={"name": "Zred_r_r+1", "support_sites": 2},
    )

    raw = {
        "Y": y_template.instantiate((0,)),
        "A": a_template.instantiate((0, 1)),
        "Z": z_template.instantiate((0, 1)),
    }
    normalized = {
        name: witness.template.normalized("operator_norm").instantiate(witness.variable_indices)
        for name, witness in raw.items()
    }
    return raw, normalized


def tower_state_for_sector(basis_configs: np.ndarray, *, length: int) -> np.ndarray:
    states, labels = spin_one_xy_scar_tower_states(
        basis_configs=basis_configs,
        length=length,
        normalize=True,
    )
    if states.shape[1] != 1:
        raise RuntimeError(f"expected one tower state in a fixed-M basis, found {labels}")
    return states[:, 0]


def tower_symmetry_sector(basis_configs: np.ndarray, scar: np.ndarray, *, length: int):
    n_raised = (TOTAL_SZ + length) // 2
    momentum_index = 0 if n_raised % 2 == 0 else length // 2

    translation = basis_permutation_from_variable_permutation(
        basis_configs,
        np.roll(np.arange(length), 1),
    )
    sector = cyclic_symmetry_sector_basis(
        translation,
        order=length,
        momentum_index=momentum_index,
        labels={"total_sz": TOTAL_SZ},
    )

    # Reflection r -> -r.  k=0 and k=pi sectors are invariant under reflection.
    reflection = basis_permutation_from_variable_permutation(
        basis_configs,
        (-np.arange(length)) % length,
    )
    reflection_value = complex(np.vdot(scar, permutation_matrix(reflection) @ scar))
    reflection_parity = 1 if reflection_value.real >= 0.0 else -1
    sector = refine_sector_by_involution(
        sector,
        reflection,
        eigenvalue=reflection_parity,
        label="reflection",
    )
    return sector, momentum_index, reflection_parity


def projected_witness_square(witness, basis_configs, sector):
    local_operator = witness.embed(basis_configs)
    q_operator = local_operator.conj().T @ local_operator
    return project_operator_to_sector(q_operator, sector)


def periodic_phase_compatible_model(*, length: int, d_z: float):
    return SpinOneXYChainModel(
        length=length,
        boundary_condition="periodic",
        j_xy=J1_MATRIX,
        d_z=d_z,
        total_sz=TOTAL_SZ,
        extra_xy_couplings=spin_one_xy_periodic_range_couplings(
            length=length,
            distance=3,
            coefficient=J3_MATRIX,
        ),
    )


RAW_WITNESSES, UNIT_WITNESSES = make_spin1_witnesses()
Y_WITNESS = RAW_WITNESSES["Y"]
A_WITNESS = RAW_WITNESSES["A"]
Z_WITNESS = RAW_WITNESSES["Z"]
Y_UNIT = UNIT_WITNESSES["Y"]
A_UNIT = UNIT_WITNESSES["A"]
Z_UNIT = UNIT_WITNESSES["Z"]

witness_norm_df = pd.DataFrame(
    [
        {
            "witness": name,
            "operator_norm_raw": RAW_WITNESSES[name].template.operator_norm,
            "Q_norm_raw": RAW_WITNESSES[name].template.q_operator_norm,
            "Delta_Q_unit": diagnose_local_channel_spectrum(
                UNIT_WITNESSES[name],
                tolerance=TOL,
            ).dark_channel_gap,
            "Q_rank_unit": diagnose_local_channel_spectrum(
                UNIT_WITNESSES[name],
                tolerance=TOL,
            ).rank,
        }
        for name in ("A", "Z", "Y")
    ]
)
display(witness_norm_df)

## A. Exact tower and the three local witnesses

We verify the exact eigenstate residual and the darkness conditions
$A_R|\mathcal S_n\rangle=Z_R|\mathcal S_n\rangle=Y_R|\mathcal S_n\rangle=0$.

In [ ]:
L_REP = 8
model_rep = periodic_phase_compatible_model(length=L_REP, d_z=0.0)
build_rep = model_rep.build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
    on_missing="raise",
)
configs_rep = basis_configs_from_build_result(build_rep)
scar_rep = tower_state_for_sector(configs_rep, length=L_REP)
support_rep = np.flatnonzero(np.abs(scar_rep) > TOL)

stability_rep = diagnose_cage_stability(
    build_rep.kinetic,
    support_rep,
    state=scar_rep,
    tolerance=TOL,
)
eigenpair_rep = diagnose_eigenpair(build_rep.hamiltonian, scar_rep)
witness_evaluations = {
    name: evaluate_local_witness_on_states(
        witness,
        basis_configs=configs_rep,
        states=scar_rep,
    )
    for name, witness in RAW_WITNESSES.items()
}

local_rows = []
for descriptor in model_rep.local_term_descriptors(operator_kind="kinetic", term_kind="bond"):
    local_matrix = model_rep.build_local_term(descriptor, build_rep, builder="optimized")
    local_rows.append(
        {
            "term": descriptor.label,
            "sites": descriptor.support_sites,
            "action_norm": float(np.linalg.norm(local_matrix @ scar_rep)),
        }
    )
local_term_df = pd.DataFrame(local_rows)

boundary_scorecard = pd.DataFrame(
    [
        {
            "L": L_REP,
            "M": TOTAL_SZ,
            "full_sector_dimension": configs_rep.shape[0],
            "support_size": support_rep.size,
            "boundary_rank": stability_rep.boundary_rank,
            "boundary_nullity": stability_rep.boundary_nullity,
            "boundary_singular_gap": stability_rep.interference_gap,
            "boundary_residual": stability_rep.state_boundary_residual,
            "internal_residual": stability_rep.state_internal_eigen_residual,
            "full_eigenpair_residual": eigenpair_rep.residual_norm,
            "A_annihilation_residual": witness_evaluations["A"].annihilation_residual,
            "Z_annihilation_residual": witness_evaluations["Z"].annihilation_residual,
            "Y_annihilation_residual": witness_evaluations["Y"].annihilation_residual,
        }
    ]
)

display(boundary_scorecard)
display(local_term_df)
boundary_scorecard.to_csv(DATA_DIR / "boundary_kernel_scorecard.csv", index=False)
local_term_df.to_csv(DATA_DIR / "local_term_annihilation.csv", index=False)
witness_norm_df.to_csv(DATA_DIR / "local_channel_spectra.csv", index=False)

The finite-size check records the tower support, the one-dimensional boundary kernel, the full eigenstate residual, and the local witness residuals.  The three positive observables are normalized to make their thermal activities directly comparable.

## T1. Generic finite-energy microcanonical comparison

We first construct the ordinary symmetry-resolved microcanonical ensemble of $H_{XY}+H_3+D\sum_r(S_r^z)^2$ at the tower energy. The same spectral pass also caches the $D=0$ matching point used later in T2.

In [ ]:
spectral_rows = []
window_sensitivity_rows = []
finite_d_window_sensitivity_rows = []
scan_cache = {}

for length in SIZES:
    t0 = time.perf_counter()
    n_raised = (TOTAL_SZ + length) // 2

    result_zero = periodic_phase_compatible_model(length=length, d_z=0.0).build(
        builder="optimized",
        basis_solver="dfs",
        sort_basis=True,
    )
    configs = basis_configs_from_build_result(result_zero)
    scar = tower_state_for_sector(configs, length=length)
    sector, momentum_index, reflection_parity = tower_symmetry_sector(
        configs,
        scar,
        length=length,
    )
    scar_sector = project_state_to_sector(scar, sector)
    scar_sector /= np.linalg.norm(scar_sector)

    y_sector = project_operator_to_sector(Y_WITNESS.embed(configs), sector)
    z_sector = project_operator_to_sector(Z_WITNESS.embed(configs), sector)
    qy_sector = projected_witness_square(Y_WITNESS, configs, sector)
    qa_sector = projected_witness_square(A_WITNESS, configs, sector)
    qz_sector = projected_witness_square(Z_WITNESS, configs, sector)

    h0_sector = project_operator_to_sector(result_zero.hamiltonian, sector)
    e0, v0 = la.eigh(h0_sector)
    y0 = eigenstate_expectations(qy_sector, v0)
    a0 = eigenstate_expectations(qa_sector, v0)
    z0 = eigenstate_expectations(qz_sector, v0)
    ymean0 = eigenstate_expectations(y_sector, v0)
    zmean0 = eigenstate_expectations(z_sector, v0)
    scar_overlap0 = np.abs(v0.conj().T @ scar_sector) ** 2
    scar_level0 = int(np.argmax(scar_overlap0))
    scar_degenerate_mask0 = np.abs(e0) <= 1.0e-8
    scar_degenerate_weight0 = float(np.sum(scar_overlap0[scar_degenerate_mask0]))

    # The eigensolver basis inside an exactly degenerate E=0 manifold is arbitrary.
    # Evaluate the known tower vector itself rather than calling the maximum-overlap
    # numerical eigenvector "the scar". This is essential for the ETH scatter plot.
    exact_scar_QY = float(np.vdot(scar_sector, qy_sector @ scar_sector).real)
    exact_scar_QA = float(np.vdot(scar_sector, qa_sector @ scar_sector).real / A_WITNESS.template.q_operator_norm)
    exact_scar_QZ = float(np.vdot(scar_sector, qz_sector @ scar_sector).real / Z_WITNESS.template.q_operator_norm)
    if max(abs(exact_scar_QY), abs(exact_scar_QA), abs(exact_scar_QZ)) > 1.0e-9:
        raise RuntimeError(
            f"exact tower is not dark at L={length}: "
            f"QY={exact_scar_QY:.3e}, QA={exact_scar_QA:.3e}, QZ={exact_scar_QZ:.3e}"
        )

    windows0 = {}
    for prefactor in WINDOW_PREFACTORS:
        plan = thermodynamic_energy_window_plan(
            volume=length,
            energy_density=0.0,
            width_prefactor=prefactor,
            local_energy_scale=J_DRAFT,
        )
        window = select_microcanonical_window_by_width(
            e0,
            target_energy=plan.target_energy,
            half_width=plan.half_width,
            degeneracy_tolerance=TOL,
        )
        indices = np.asarray(window.indices, dtype=np.int64)
        windows0[prefactor] = (plan, window, indices)
        window_sensitivity_rows.append(
            {
                "case": "D0",
                "L": length,
                "window_prefactor": prefactor,
                "target_energy": plan.target_energy,
                "requested_half_width": plan.half_width,
                "energy_density_half_width": plan.energy_density_half_width,
                "n_states": window.n_states,
                "center_offset": window.center_offset,
                "tau_Y": float(np.mean(y0[indices])),
                "tau_A_normalized": float(np.mean(a0[indices]) / A_WITNESS.template.q_operator_norm),
                "tau_Z_normalized": float(np.mean(z0[indices]) / Z_WITNESS.template.q_operator_norm),
                "mean_Y": float(np.mean(ymean0[indices])),
                "mean_Z_normalized": float(np.mean(zmean0[indices]) / np.sqrt(Z_WITNESS.template.q_operator_norm)),
            }
        )
    plan0, window0, idx0 = windows0[PRIMARY_WINDOW_PREFACTOR]
    smooth0 = gaussian_spectral_filter(
        e0,
        target_energy=0.0,
        sigma=SMOOTH_SIGMA_PREFACTOR * J_DRAFT * np.sqrt(length),
    )
    smooth_weights0 = np.asarray(smooth0.weights, dtype=np.float64)

    # Finite D uses the same subextensive-width rule, centered at E_scar=D L.
    result_d = periodic_phase_compatible_model(length=length, d_z=D_THERMAL).build(
        builder="optimized",
        basis_solver="dfs",
        sort_basis=True,
    )
    np.testing.assert_array_equal(result_d.basis.states, result_zero.basis.states)
    hd_sector = project_operator_to_sector(result_d.hamiltonian, sector)
    ed, vd = la.eigh(hd_sector)
    yd = eigenstate_expectations(qy_sector, vd)
    ad = eigenstate_expectations(qa_sector, vd)
    zd = eigenstate_expectations(qz_sector, vd)
    ymean_d = eigenstate_expectations(y_sector, vd)
    zmean_d = eigenstate_expectations(z_sector, vd)
    scar_energy = D_THERMAL * length
    scar_overlap_d = np.abs(vd.conj().T @ scar_sector) ** 2
    scar_level_d = int(np.argmax(scar_overlap_d))
    windows_d = {}
    for prefactor in WINDOW_PREFACTORS:
        plan = thermodynamic_energy_window_plan(
            volume=length,
            energy_density=D_THERMAL,
            width_prefactor=prefactor,
            local_energy_scale=J_DRAFT,
        )
        window = select_microcanonical_window_by_width(
            ed,
            target_energy=scar_energy,
            half_width=plan.half_width,
            degeneracy_tolerance=TOL,
        )
        indices = np.asarray(window.indices, dtype=np.int64)
        windows_d[prefactor] = (plan, window, indices)
        finite_d_window_sensitivity_rows.append(
            {
                "L": length,
                "D": D_THERMAL,
                "window_prefactor": prefactor,
                "target_energy": scar_energy,
                "target_energy_density": D_THERMAL,
                "requested_half_width": plan.half_width,
                "energy_density_half_width": plan.energy_density_half_width,
                "n_states": window.n_states,
                "center_offset": window.center_offset,
                "tau_Y": float(np.mean(yd[indices])),
                "tau_A_normalized": float(np.mean(ad[indices]) / A_WITNESS.template.q_operator_norm),
                "tau_Z_normalized": float(np.mean(zd[indices]) / Z_WITNESS.template.q_operator_norm),
                "mean_Y": float(np.mean(ymean_d[indices])),
                "var_Y": float(np.mean(yd[indices]) - np.mean(ymean_d[indices]) ** 2),
                "mean_Z_normalized": float(np.mean(zmean_d[indices]) / np.sqrt(Z_WITNESS.template.q_operator_norm)),
                "var_Z_normalized": float((np.mean(zd[indices]) - np.mean(zmean_d[indices]) ** 2) / Z_WITNESS.template.q_operator_norm),
            }
        )
    plan_d, windowd, idxd = windows_d[PRIMARY_WINDOW_PREFACTOR]
    smoothd = gaussian_spectral_filter(
        ed,
        target_energy=scar_energy,
        sigma=SMOOTH_SIGMA_PREFACTOR * J_DRAFT * np.sqrt(length),
    )
    smooth_weights_d = np.asarray(smoothd.weights, dtype=np.float64)

    gap = adjacent_gap_ratio_report(
        e0,
        trim_fraction=0.10,
        degeneracy_tolerance=1.0e-8,
    )
    exact = spin_one_xy_tower_thermal_activities(
        length=length,
        total_sz=TOTAL_SZ,
        xy_matrix_element=J1_MATRIX,
    )
    residual_d = diagnose_eigenpair(result_d.hamiltonian, scar)
    exact_scar_QY_d = float(np.vdot(scar_sector, qy_sector @ scar_sector).real)
    exact_scar_QA_d = float(np.vdot(scar_sector, qa_sector @ scar_sector).real / A_WITNESS.template.q_operator_norm)
    exact_scar_QZ_d = float(np.vdot(scar_sector, qz_sector @ scar_sector).real / Z_WITNESS.template.q_operator_norm)
    if max(abs(exact_scar_QY_d), abs(exact_scar_QA_d), abs(exact_scar_QZ_d)) > 1.0e-9:
        raise RuntimeError(
            f"finite-D exact tower is not dark at L={length}: "
            f"QY={exact_scar_QY_d:.3e}, QA={exact_scar_QA_d:.3e}, QZ={exact_scar_QZ_d:.3e}"
        )

    spectral_rows.append(
        {
            "L": length,
            "M": TOTAL_SZ,
            "n_raised": n_raised,
            "full_M_sector_dimension": configs.shape[0],
            "momentum_index": momentum_index,
            "momentum_over_pi": 2.0 * momentum_index / length,
            "reflection_parity": reflection_parity,
            "resolved_sector_dimension": sector.sector_dimension,
            "J3_over_J": J3_OVER_J,
            "D0_scar_max_single_vector_overlap": scar_overlap0[scar_level0],
            "D0_scar_degenerate_subspace_weight": scar_degenerate_weight0,
            "D0_scar_degenerate_level_count": int(np.sum(scar_degenerate_mask0)),
            "D0_scar_level_energy": e0[scar_level0],
            "D0_exact_scar_QY": exact_scar_QY,
            "D0_exact_scar_QA_normalized": exact_scar_QA,
            "D0_exact_scar_QZ_normalized": exact_scar_QZ,
            "D0_window_requested_half_width": plan0.half_width,
            "D0_window_energy_density_half_width": plan0.energy_density_half_width,
            "D0_window_actual_half_width": window0.half_width,
            "D0_window_state_count": window0.n_states,
            "D0_window_center_offset": window0.center_offset,
            "D0_microcanonical_Y2": float(np.mean(y0[idx0])),
            "D0_microcanonical_A2_normalized": float(np.mean(a0[idx0]) / A_WITNESS.template.q_operator_norm),
            "D0_microcanonical_Z2_normalized": float(np.mean(z0[idx0]) / Z_WITNESS.template.q_operator_norm),
            "D0_microcanonical_Y_mean": float(np.mean(ymean0[idx0])),
            "D0_microcanonical_Y_variance": float(np.mean(y0[idx0]) - np.mean(ymean0[idx0]) ** 2),
            "D0_microcanonical_Z_mean_normalized": float(np.mean(zmean0[idx0]) / np.sqrt(Z_WITNESS.template.q_operator_norm)),
            "D0_microcanonical_Z_variance_normalized": float((np.mean(z0[idx0]) - np.mean(zmean0[idx0]) ** 2) / Z_WITNESS.template.q_operator_norm),
            "D0_smooth_Y2": float(np.dot(smooth_weights0, y0)),
            "D0_smooth_A2_normalized": float(np.dot(smooth_weights0, a0) / A_WITNESS.template.q_operator_norm),
            "D0_smooth_Z2_normalized": float(np.dot(smooth_weights0, z0) / Z_WITNESS.template.q_operator_norm),
            "D0_smooth_effective_state_count": smooth0.effective_state_count,
            "exact_fixed_M_Y2": exact.y2_activity,
            "exact_fixed_M_A2_normalized": exact.directed_q_activity / A_WITNESS.template.q_operator_norm,
            "exact_fixed_M_Z2_normalized": exact.z2_activity / Z_WITNESS.template.q_operator_norm,
            "finiteD_D": D_THERMAL,
            "finiteD_scar_energy": scar_energy,
            "finiteD_scar_level_energy": ed[scar_level_d],
            "finiteD_scar_overlap": scar_overlap_d[scar_level_d],
            "finiteD_scar_residual": residual_d.residual_norm,
            "finiteD_exact_scar_QY": exact_scar_QY_d,
            "finiteD_exact_scar_QA_normalized": exact_scar_QA_d,
            "finiteD_exact_scar_QZ_normalized": exact_scar_QZ_d,
            "finiteD_window_requested_half_width": plan_d.half_width,
            "finiteD_window_energy_density_half_width": plan_d.energy_density_half_width,
            "finiteD_window_actual_half_width": windowd.half_width,
            "finiteD_window_state_count": windowd.n_states,
            "finiteD_window_center_offset": windowd.center_offset,
            "finiteD_microcanonical_Y2": float(np.mean(yd[idxd])),
            "finiteD_microcanonical_A2_normalized": float(np.mean(ad[idxd]) / A_WITNESS.template.q_operator_norm),
            "finiteD_microcanonical_Z2_normalized": float(np.mean(zd[idxd]) / Z_WITNESS.template.q_operator_norm),
            "finiteD_microcanonical_Y_mean": float(np.mean(ymean_d[idxd])),
            "finiteD_microcanonical_Y_variance": float(np.mean(yd[idxd]) - np.mean(ymean_d[idxd]) ** 2),
            "finiteD_microcanonical_Z_mean_normalized": float(np.mean(zmean_d[idxd]) / np.sqrt(Z_WITNESS.template.q_operator_norm)),
            "finiteD_microcanonical_Z_variance_normalized": float((np.mean(zd[idxd]) - np.mean(zmean_d[idxd]) ** 2) / Z_WITNESS.template.q_operator_norm),
            "finiteD_smooth_A2_normalized": float(np.dot(smooth_weights_d, ad) / A_WITNESS.template.q_operator_norm),
            "finiteD_sharp_smooth_A_difference": float(abs(np.mean(ad[idxd]) / A_WITNESS.template.q_operator_norm - np.dot(smooth_weights_d, ad) / A_WITNESS.template.q_operator_norm)),
            "mean_gap_ratio_D0": gap.mean_ratio,
            "gap_ratio_count_D0": len(gap.ratios),
            "runtime_seconds": time.perf_counter() - t0,
        }
    )
    scan_cache[length] = {
        "configs": configs,
        "scar": scar,
        "sector": sector,
        "energies_D0": e0,
        "vectors_D0": v0,
        "Y2_D0": y0,
        "A2_D0": a0,
        "Z2_D0": z0,
        "Y_D0": ymean0,
        "Z_D0": zmean0,
        "scar_level_D0": scar_level0,
        "exact_scar_QY": exact_scar_QY,
        "exact_scar_QA_normalized": exact_scar_QA,
        "exact_scar_QZ_normalized": exact_scar_QZ,
        "window_D0": window0,
        "window_plan_D0": plan0,
        "energies_D": ed,
        "vectors_D": vd,
        "Y2_D": yd,
        "A2_D": ad,
        "Z2_D": zd,
        "Y_D": ymean_d,
        "Z_D": zmean_d,
        "scar_level_D": scar_level_d,
        "exact_scar_QY_D": exact_scar_QY_d,
        "exact_scar_QA_normalized_D": exact_scar_QA_d,
        "exact_scar_QZ_normalized_D": exact_scar_QZ_d,
        "window_D": windowd,
        "window_plan_D": plan_d,
        "gap_report": gap,
    }

spectral_df = pd.DataFrame(spectral_rows)
window_sensitivity_df = pd.DataFrame(window_sensitivity_rows)
finite_d_window_sensitivity_df = pd.DataFrame(finite_d_window_sensitivity_rows)
display(spectral_df)
spectral_df.to_csv(DATA_DIR / "symmetry_resolved_spectral_evidence.csv", index=False)
window_sensitivity_df.to_csv(DATA_DIR / "D0_microcanonical_window_sensitivity.csv", index=False)
finite_d_window_sensitivity_df.to_csv(DATA_DIR / "finiteD_microcanonical_window_sensitivity.csv", index=False)

### T1a. Finite-$D$ ETH scatter, window scaling, and Hermitian resolution

The finite-$D$ point is the primary same-Hamiltonian energy-resolved test. The exact tower vector is plotted separately at zero witness value, and the shaded interval is the actual degeneracy-completed microcanonical window.

In [ ]:
# Dedicated finite-D ETH scatter, window sensitivity, and Hermitian mean/variance.
largest_L_D = max(SIZES)
finite_d = scan_cache[largest_L_D]
energies_d = finite_d["energies_D"]
qyd = finite_d["Y2_D"]
qad = finite_d["A2_D"] / A_WITNESS.template.q_operator_norm
qzd = finite_d["Z2_D"] / Z_WITNESS.template.q_operator_norm
window_d = finite_d["window_D"]
plan_d = finite_d["window_plan_D"]

finite_d_scatter_df = pd.DataFrame(
    {
        "energy": energies_d,
        "energy_density": energies_d / largest_L_D,
        "QY": qyd,
        "QA_normalized": qad,
        "QZ_normalized": qzd,
        "is_microcanonical": np.isin(np.arange(energies_d.size), np.asarray(window_d.indices, dtype=np.int64)),
    }
)
finite_d_scatter_df.to_csv(DATA_DIR / "finiteD_eth_scatter_Lmax.csv", index=False)

center_density_d = D_THERMAL
half_density_d = float(window_d.half_width) / largest_L_D
fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
ax.axvspan(
    center_density_d - half_density_d,
    center_density_d + half_density_d,
    color="0.5",
    alpha=0.10,
    label="microcanonical window",
    zorder=0,
)
ax.axvline(center_density_d, color="0.45", linestyle="--", linewidth=0.8)
ax.scatter(finite_d_scatter_df["energy_density"], finite_d_scatter_df["QY"], s=12, alpha=0.65, label=r"$Q^Y_r$")
ax.scatter(finite_d_scatter_df["energy_density"], finite_d_scatter_df["QA_normalized"], s=12, alpha=0.65, label=r"$Q^A_{r,r+1}/(8J^2)$")
ax.scatter(finite_d_scatter_df["energy_density"], finite_d_scatter_df["QZ_normalized"], s=12, alpha=0.65, label=r"$Q^Z_{r,r+1}/(8J^2)$")
ax.scatter([center_density_d], [0.0], marker="*", s=90, edgecolors="black", linewidths=0.5, label="exact tower", zorder=5)
ax.set_xlabel(r"Energy density $e=E/L$")
ax.set_ylabel("Normalized local activity")
ax.legend(loc="upper right")
ax.grid(alpha=0.25)
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_finiteD_eth_scatter")
plt.show()

finite_d_central = finite_d_window_sensitivity_df[
    np.isclose(finite_d_window_sensitivity_df["window_prefactor"], PRIMARY_WINDOW_PREFACTOR)
].copy()
by_L_D = finite_d_window_sensitivity_df.groupby("L")
fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
for column, label, marker in (
    ("tau_Y", r"$Q^Y_r$", "o"),
    ("tau_A_normalized", r"$Q^A_{r,r+1}/(8J^2)$", "s"),
    ("tau_Z_normalized", r"$Q^Z_{r,r+1}/(8J^2)$", "^"),
):
    lows = by_L_D[column].min().reindex(finite_d_central["L"]).to_numpy()
    highs = by_L_D[column].max().reindex(finite_d_central["L"]).to_numpy()
    values = finite_d_central[column].to_numpy()
    ax.errorbar(
        finite_d_central["L"],
        values,
        yerr=np.vstack([values - lows, highs - values]),
        marker=marker,
        capsize=3,
        label=label,
    )
ax.set_xlabel(r"System size $L$")
ax.set_ylabel("Finite-$D$ microcanonical activity")
ax.legend(loc="upper right")
ax.grid(alpha=0.25)
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_finiteD_microcanonical_convergence")
plt.show()

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
ax.plot(finite_d_central["L"], -finite_d_central["mean_Y"], marker="o", label=r"$-\langle Y_r\rangle_{\rm mc}$")
ax.plot(finite_d_central["L"], finite_d_central["var_Y"], marker="s", label=r"${\rm Var}_{\rm mc}(Y_r)$")
ax.plot(finite_d_central["L"], finite_d_central["mean_Z_normalized"], marker="^", label=r"$\langle Z\rangle_{\rm mc}/\sqrt{8J^2}$")
ax.plot(finite_d_central["L"], finite_d_central["var_Z_normalized"], marker="v", label=r"${\rm Var}_{\rm mc}(Z)/(8J^2)$")
ax.set_xlabel(r"System size $L$")
ax.set_ylabel("Finite-$D$ mean or variance")
ax.legend(loc="upper right")
ax.grid(alpha=0.25)
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_finiteD_mean_variance")
plt.show()


In [ ]:
# The draft-oriented figures are saved in DATA_DIR by the preceding cell.
print("saved D=0 ETH figures in", DATA_DIR)

### T1b. Background concentration beyond the selected witnesses

A charge-conserving two-site Hermitian basis tests whether typical resolved-sector eigenstates concentrate locally, rather than checking only $A_R$, $Z_R$, and $Y_R$.

In [ ]:
def charge_conserving_two_site_basis():
    patterns = tuple((a, b) for a in (-1, 0, 1) for b in (-1, 0, 1))
    groups = {}
    for index, pattern in enumerate(patterns):
        groups.setdefault(sum(pattern), []).append(index)
    operators = []
    for charge, indices in sorted(groups.items()):
        for i in indices:
            matrix = np.zeros((9, 9), dtype=np.complex128)
            matrix[i, i] = 1.0
            operators.append((f"q{charge}_diag_{i}", matrix))
        for offset, i in enumerate(indices):
            for j in indices[offset + 1:]:
                sym = np.zeros((9, 9), dtype=np.complex128)
                sym[i, j] = sym[j, i] = 1.0
                asym = np.zeros((9, 9), dtype=np.complex128)
                asym[i, j] = -1.0j
                asym[j, i] = 1.0j
                operators.append((f"q{charge}_sym_{i}_{j}", sym))
                operators.append((f"q{charge}_asym_{i}_{j}", asym))
    return patterns, operators

background_rows = []
if RUN_BACKGROUND_CONCENTRATION:
    pair_patterns, pair_basis = charge_conserving_two_site_basis()
    for length in SIZES:
        cached = scan_cache[length]
        for case, energies_key, vectors_key, window_key in (
            ("D0_J3", "energies_D0", "vectors_D0", "window_D0"),
            ("finiteD_J3", "energies_D", "vectors_D", "window_D"),
        ):
            configs = cached["configs"]
            sector = cached["sector"]
            energies = cached[energies_key]
            vectors = cached[vectors_key]
            indices = np.asarray(cached[window_key].indices, dtype=np.int64)
            for name, matrix in pair_basis:
                template = LocalWitnessTemplate(
                    pattern_key=(),
                    local_patterns=pair_patterns,
                    local_operator=matrix,
                    metadata={"name": name},
                ).normalized("operator_norm")
                operator = project_operator_to_sector(template.instantiate((0, 1)).embed(configs), sector)
                diagnostic = degeneracy_resolved_concentration(
                    energies,
                    vectors,
                    operator,
                    indices,
                    energy_tolerance=1.0e-9,
                )
                background_rows.append({"case": case, "L": int(length), "operator": name, **diagnostic})
background_concentration_df = pd.DataFrame(background_rows)
background_concentration_df.to_csv(DATA_DIR / "spin1_xy_background_concentration.csv", index=False)
if not background_concentration_df.empty:
    display(
        background_concentration_df.groupby(["case", "L"])[
            ["basis_independent_std", "p90_abs_deviation", "max_abs_deviation"]
        ].agg(["median", "max"])
    )

    envelope = background_concentration_df.groupby(["case", "L"]).agg(
        median_std=("basis_independent_std", "median"),
        max_std=("basis_independent_std", "max"),
        max_p90=("p90_abs_deviation", "max"),
    ).reset_index()
    fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
    for case, frame in envelope.groupby("case"):
        ax.plot(frame["L"], frame["median_std"], marker="o", label=case + " median")
        ax.plot(frame["L"], frame["max_std"], marker="s", linestyle="--", label=case + " max")
    ax.set_xlabel(r"System size $L$")
    ax.set_ylabel("Window EEV spread")
    ax.legend(frameon=False, fontsize=8)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    save_spin_figure(fig, "spin1_xy_background_concentration")
    plt.show()

## T2. Matched $D=0$, $\beta=0$ extension

At $D=0$ the tower lies at the resolved-sector infinite-temperature energy. Exact fixed-$M$ traces provide large-size targets, but are used only after the finite-size microcanonical--normalized-trace overlap is exposed explicitly.

In [ ]:
formula_rows = []
for length in COUNTING_LENGTHS:
    exact = spin_one_xy_tower_thermal_activities(
        length=length,
        total_sz=TOTAL_SZ,
        xy_matrix_element=J1_MATRIX,
    )
    formula_rows.append(exact.to_summary_dict())
formula_df = pd.DataFrame(formula_rows)

# Independent direct traces in the qlinks fixed-M basis for ED-accessible sizes.
direct_rows = []
for length in (4, 6, 8, 10):
    result = SpinOneXYChainModel(
        length=length,
        boundary_condition="periodic",
        j_xy=0.0,
        total_sz=TOTAL_SZ,
    ).build(builder="optimized", basis_solver="dfs", sort_basis=True)
    configs = basis_configs_from_build_result(result)
    evaluations = {
        name: evaluate_local_witness_on_diagonal_ensemble(
            witness,
            basis_configs=configs,
        )
        for name, witness in RAW_WITNESSES.items()
    }
    direct_rows.append(
        {
            "length": length,
            "basis_dimension": configs.shape[0],
            "Y2_direct_trace": evaluations["Y"].expectation,
            "A2_direct_trace": evaluations["A"].expectation,
            "Z2_direct_trace": evaluations["Z"].expectation,
            "A2_direct_normalized": evaluations["A"].normalized_expectation,
            "Z2_direct_normalized": evaluations["Z"].normalized_expectation,
        }
    )
direct_df = pd.DataFrame(direct_rows)
activity_df = formula_df.merge(direct_df, how="left", on="length")
activity_df["Y2_direct_minus_formula"] = activity_df["Y2_direct_trace"] - activity_df["y2_activity"]
activity_df["A2_direct_minus_formula"] = activity_df["A2_direct_trace"] - activity_df["directed_q_activity"]
activity_df["Z2_direct_minus_formula"] = activity_df["Z2_direct_trace"] - activity_df["z2_activity"]

display(activity_df.head(8))
activity_df.to_csv(DATA_DIR / "exact_fixed_M_activities.csv", index=False)

# Thermodynamic asymptotes for the fixed-M sequence TOTAL_SZ=-2, where q=M/L -> 0.
p0_infty = 1.0 / 3.0
y2_infty = p0_infty
a2_infty = 2.0 * abs(J1_MATRIX) ** 2 * p0_infty**2
z2_infty = 4.0 * abs(J1_MATRIX) ** 2 * p0_infty**2
a2_infty_normalized = a2_infty / A_WITNESS.template.q_operator_norm
z2_infty_normalized = z2_infty / Z_WITNESS.template.q_operator_norm

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
ax.plot(activity_df["length"], activity_df["y2_activity"], marker="o", label=r"$Q^Y_r$")
ax.plot(
    activity_df["length"],
    activity_df["A2_direct_normalized"],
    marker="s",
    label=r"$Q^A_{r,r+1}/(8J^2)$",
)
ax.plot(
    activity_df["length"],
    activity_df["Z2_direct_normalized"],
    marker="^",
    label=r"$Q^Z_{r,r+1}/(8J^2)$",
)
ax.axhline(y2_infty, linestyle="--", linewidth=0.8)
ax.axhline(a2_infty_normalized, linestyle="--", linewidth=0.8)
ax.axhline(z2_infty_normalized, linestyle="--", linewidth=0.8)
ax.set_xlabel(r"System size $L$")
ax.set_ylabel("Normalized thermal activity")
ax.legend(loc="upper right", frameon=False)
ax.grid(alpha=0.3)
fig.tight_layout()
save_spin_figure(fig, "exact_fixed_M_three_witness_activities")
plt.show()


### T2a. $D=0$ ETH scatter and microcanonical convergence

In [ ]:
largest_L = max(SIZES)
largest = scan_cache[largest_L]
energies = largest["energies_D0"]
y_values = largest["Y2_D0"]
a_values = largest["A2_D0"] / A_WITNESS.template.q_operator_norm
z_values = largest["Z2_D0"] / Z_WITNESS.template.q_operator_norm
scar_level = largest["scar_level_D0"]

scatter_df = pd.DataFrame(
    {
        "energy": energies,
        "energy_density": energies / largest_L,
        "QY": y_values,
        "QA_normalized": a_values,
        "QZ_normalized": z_values,
        # This only identifies the arbitrary eigensolver vector with maximum
        # tower overlap inside the degenerate zero-energy manifold.
        "is_max_overlap_vector": np.arange(energies.size) == scar_level,
        "is_microcanonical": np.isin(
            np.arange(energies.size),
            np.asarray(largest["window_D0"].indices, dtype=np.int64),
        ),
    }
)
exact_scar_scatter = pd.DataFrame(
    {
        "energy": [0.0],
        "energy_density": [0.0],
        "QY": [largest["exact_scar_QY"]],
        "QA_normalized": [largest["exact_scar_QA_normalized"]],
        "QZ_normalized": [largest["exact_scar_QZ_normalized"]],
    }
)
scatter_df.to_csv(DATA_DIR / "eth_scatter_Lmax_D0.csv", index=False)
exact_scar_scatter.to_csv(DATA_DIR / "eth_scatter_Lmax_exact_tower.csv", index=False)

central = window_sensitivity_df[np.isclose(window_sensitivity_df["window_prefactor"], PRIMARY_WINDOW_PREFACTOR)].copy()


def plot_figure3a(ax, *, length: int | None = None):
    if length is None:
        length = largest_L
    cached = scan_cache[length]
    if length == largest_L:
        subset = scatter_df
    else:
        window_indices = np.asarray(cached["window_D0"].indices, dtype=np.int64)
        subset = pd.DataFrame(
            {
                "energy_density": cached["energies_D0"] / length,
                "QY": cached["Y2_D0"],
                "QA_normalized": cached["A2_D0"] / A_WITNESS.template.q_operator_norm,
                "QZ_normalized": cached["Z2_D0"] / Z_WITNESS.template.q_operator_norm,
                "is_microcanonical": np.isin(np.arange(cached["energies_D0"].size), window_indices),
            }
        )

    # Shade the actual selected energy window. Degeneracy completion may make
    # this slightly wider than the requested c J sqrt(L) interval.
    half_width_density = float(cached["window_D0"].half_width) / float(length)
    ax.axvspan(
        -half_width_density,
        half_width_density,
        alpha=0.10,
        color="0.5",
        label="microcanonical window",
        zorder=0,
    )
    ax.axvline(0.0, linewidth=0.7, linestyle="--", color="0.45", zorder=1)

    ax.scatter(subset["energy_density"], subset["QY"], s=10, alpha=0.65, label=r"$Q^Y_r$")
    ax.scatter(subset["energy_density"], subset["QA_normalized"], s=10, alpha=0.65, label=r"$Q^A_{r,r+1}/(8J^2)$")
    ax.scatter(subset["energy_density"], subset["QZ_normalized"], s=10, alpha=0.65, label=r"$Q^Z_{r,r+1}/(8J^2)$")

    # Plot the analytically known tower vector itself. A numerical diagonalizer
    # may return arbitrary mixtures inside the degenerate E=0 manifold, whose
    # witness expectation need not vanish even though the exact tower is dark.
    scar_values = (
        cached["exact_scar_QY"],
        cached["exact_scar_QA_normalized"],
        cached["exact_scar_QZ_normalized"],
    )
    ax.scatter([0.0], [scar_values[0]], marker="*", s=80, edgecolors="black", linewidths=0.5, label="exact tower", zorder=5)
    ax.set_xlabel(r"Energy density $e=E/L$")
    ax.set_ylabel("Normalized local activity")
    ax.grid(alpha=0.3)
    return ax


def plot_figure3b(ax):
    by_L = window_sensitivity_df.groupby("L")
    for column, label, asymptote in (
        ("tau_Y", r"$Q^Y_r$", 1.0 / 3.0),
        ("tau_A_normalized", r"$Q^A_{r,r+1}/(8J^2)$", 1.0 / 9.0),
        ("tau_Z_normalized", r"$Q^Z_{r,r+1}/(8J^2)$", 2.0 / 9.0),
    ):
        lows = by_L[column].min().reindex(central["L"]).to_numpy()
        highs = by_L[column].max().reindex(central["L"]).to_numpy()
        values = central[column].to_numpy()
        ax.errorbar(
            central["L"],
            values,
            yerr=np.vstack([values - lows, highs - values]),
            marker="o",
            capsize=3,
            label=label,
        )
        ax.axhline(asymptote, linestyle="--", linewidth=0.8)
    ax.set_xlabel(r"System size $L$")
    ax.set_ylabel("Normalized thermal activity")
    ax.grid(alpha=0.3)
    return ax


def plot_figure3c(ax):
    ax.plot(spectral_df["L"], -spectral_df["D0_microcanonical_Y_mean"], marker="o", label=r"$-\langle Y_r\rangle_{\rm mc}$")
    ax.plot(spectral_df["L"], spectral_df["D0_microcanonical_Y_variance"], marker="s", label=r"${\rm Var}_{\rm mc}(Y_r)$")
    ax.plot(spectral_df["L"], spectral_df["D0_microcanonical_Z_mean_normalized"], marker="^", label=r"$\langle Z\rangle_{\rm mc}/\sqrt{8J^2}$")
    ax.plot(spectral_df["L"], spectral_df["D0_microcanonical_Z_variance_normalized"], marker="v", label=r"${\rm Var}_{\rm mc}(Z)/(8J^2)$")
    ax.set_xlabel(r"System size $L$")
    ax.set_ylabel("Mean or variance")
    ax.grid(alpha=0.3)
    return ax


fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
plot_figure3a(ax)
ax.legend(loc="upper right", frameon=False)
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_eth_scatter", aliases=("spin1_xy_D0_eth_scatter",))
plt.show()

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
plot_figure3b(ax)
ax.legend(loc="upper right", frameon=False)
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_microcanonical_convergence", aliases=("spin1_xy_D0_microcanonical_convergence",))
plt.show()

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
plot_figure3c(ax)
ax.legend(loc="upper right", frameon=False)
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_mean_variance", aliases=("spin1_xy_D0_mean_variance",))
plt.show()

display(scatter_df.iloc[max(0, scar_level - 3): scar_level + 4])


### T2b. Finite-size microcanonical--$\beta=0$ overlap

For every ED size, this table records the cage and normalized resolved-sector trace energy densities, the same three witness values in the microcanonical and $\beta=0$ ensembles, and their absolute differences.

In [ ]:
beta0_overlap_rows = []
for row in spectral_df.itertuples(index=False):
    length = int(row.L)
    cached = scan_cache[length]
    energies = np.asarray(cached["energies_D0"], dtype=float)
    beta0_values = {
        "Y": float(np.mean(cached["Y2_D0"])),
        "A": float(np.mean(cached["A2_D0"]) / A_WITNESS.template.q_operator_norm),
        "Z": float(np.mean(cached["Z2_D0"]) / Z_WITNESS.template.q_operator_norm),
    }
    mc_values = {
        "Y": float(row.D0_microcanonical_Y2),
        "A": float(row.D0_microcanonical_A2_normalized),
        "Z": float(row.D0_microcanonical_Z2_normalized),
    }
    beta0_overlap_rows.append(
        {
            "L": length,
            "resolved_sector_dimension": int(row.resolved_sector_dimension),
            "cage_energy_density": 0.0,
            "beta0_trace_energy_density": float(np.mean(energies) / length),
            "energy_density_mismatch": float(abs(np.mean(energies) / length)),
            "window_energy_density_half_width": float(row.D0_window_energy_density_half_width),
            "window_state_count": int(row.D0_window_state_count),
            "tau_A_microcanonical": mc_values["A"],
            "tau_A_beta0": beta0_values["A"],
            "delta_A": abs(mc_values["A"] - beta0_values["A"]),
            "tau_Z_microcanonical": mc_values["Z"],
            "tau_Z_beta0": beta0_values["Z"],
            "delta_Z": abs(mc_values["Z"] - beta0_values["Z"]),
            "tau_Y_microcanonical": mc_values["Y"],
            "tau_Y_beta0": beta0_values["Y"],
            "delta_Y": abs(mc_values["Y"] - beta0_values["Y"]),
        }
    )
spin_beta0_overlap_df = pd.DataFrame(beta0_overlap_rows)
spin_beta0_overlap_df.to_csv(DATA_DIR / "spin1_xy_beta0_ensemble_overlap.csv", index=False)
display(spin_beta0_overlap_df)

fig = plt.figure(figsize=PRX_TWO_PANEL_FIGSIZE)
grid = fig.add_gridspec(1, 2, wspace=0.32)
ax_e = fig.add_subplot(grid[0, 0])
ax_q = fig.add_subplot(grid[0, 1])
ax_e.plot(spin_beta0_overlap_df["L"], spin_beta0_overlap_df["energy_density_mismatch"], marker="o")
ax_e.set_xlabel(r"System size $L$")
ax_e.set_ylabel(r"$|e_{\psi,L}-e_{\beta=0,L}|$")
ax_e.grid(alpha=0.25)
add_panel_label(ax_e, "(a)")
for column, label, marker in (("delta_A", r"$A_R$", "o"), ("delta_Z", r"$Z_R$", "s"), ("delta_Y", r"$Y_R$", "^")):
    ax_q.plot(spin_beta0_overlap_df["L"], spin_beta0_overlap_df[column], marker=marker, label=label)
ax_q.set_xlabel(r"System size $L$")
ax_q.set_ylabel(r"$|\tau_Q^{\rm mc}-\tau_Q^{\rm can}(0)|$")
ax_q.legend(loc="upper right", frameon=False)
ax_q.grid(alpha=0.25)
add_panel_label(ax_q, "(b)")
fig.subplots_adjust(left=0.10, right=0.985, bottom=0.18, top=0.96, wspace=0.32)
save_spin_figure(fig, "spin1_xy_beta0_ensemble_overlap")
plt.show()

## T3. Finite-temperature canonical--microcanonical comparison

In [ ]:
finite_d_beta_rows = []
for length in SIZES:
    cached = scan_cache[length]
    match = canonical_beta_match(
        cached["energies_D"],
        D_THERMAL * length,
        tolerance=1.0e-12,
    )
    weights = np.asarray(match.pop("weights"), dtype=np.float64)
    finite_d_beta_rows.append(
        {
            "L": int(length),
            "D": float(D_THERMAL),
            "beta_L": float(match["beta"]),
            "target_energy": float(match["target_energy"]),
            "matched_energy": float(match["matched_energy"]),
            "energy_residual": float(match["energy_residual"]),
            "canonical_effective_state_count": float(match["effective_state_count"]),
            "canonical_tau_Y": float(np.dot(weights, cached["Y2_D"])),
            "canonical_tau_A_normalized": float(np.dot(weights, cached["A2_D"]) / A_WITNESS.template.q_operator_norm),
            "canonical_tau_Z_normalized": float(np.dot(weights, cached["Z2_D"]) / Z_WITNESS.template.q_operator_norm),
            "microcanonical_tau_Y": float(spectral_df.loc[spectral_df["L"] == length, "finiteD_microcanonical_Y2"].iloc[0]),
            "microcanonical_tau_A_normalized": float(spectral_df.loc[spectral_df["L"] == length, "finiteD_microcanonical_A2_normalized"].iloc[0]),
            "microcanonical_tau_Z_normalized": float(spectral_df.loc[spectral_df["L"] == length, "finiteD_microcanonical_Z2_normalized"].iloc[0]),
        }
    )
finite_d_beta_df = pd.DataFrame(finite_d_beta_rows)
finite_d_beta_df.to_csv(DATA_DIR / "finiteD_matched_beta.csv", index=False)
display(finite_d_beta_df)

fig = plt.figure(figsize=PRX_TWO_PANEL_FIGSIZE)
grid = fig.add_gridspec(1, 2, wspace=0.30)
ax0 = fig.add_subplot(grid[0, 0])
ax1 = fig.add_subplot(grid[0, 1])
ax0.plot(finite_d_beta_df["L"], finite_d_beta_df["beta_L"], marker="o")
ax0.axhline(0.0, linestyle="--", linewidth=0.8)
ax0.set_xlabel(r"System size $L$")
ax0.set_ylabel(r"Matched inverse temperature $\beta_L$")
ax0.grid(alpha=0.25)
for mc_col, can_col, label, marker in (
    ("microcanonical_tau_Y", "canonical_tau_Y", r"$Q^Y$", "o"),
    ("microcanonical_tau_A_normalized", "canonical_tau_A_normalized", r"$Q^A/(8J^2)$", "s"),
    ("microcanonical_tau_Z_normalized", "canonical_tau_Z_normalized", r"$Q^Z/(8J^2)$", "^"),
):
    ax1.plot(finite_d_beta_df["L"], finite_d_beta_df[mc_col], marker=marker, label=label + " MC")
    ax1.plot(finite_d_beta_df["L"], finite_d_beta_df[can_col], marker=marker, linestyle="--", label=label + " canonical")
ax1.set_xlabel(r"System size $L$")
ax1.set_ylabel("Normalized activity")
ax1.legend(frameon=False, fontsize=8)
ax1.grid(alpha=0.25)
for ax, panel in ((ax0, "(a)"), (ax1, "(b)")):
    ax.text(0.02, 0.98, panel, transform=ax.transAxes, ha="left", va="top")
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_finiteD_matched_beta")
plt.show()

## C7. Projector-deletion control in the undeformed spectrum

In [ ]:
protocol_m_rows = []
if RUN_PROTOCOL_M:
    for length in SIZES:
        undeformed = SpinOneXYChainModel(
            length=length,
            boundary_condition="periodic",
            j_xy=J1_MATRIX,
            d_z=0.0,
            total_sz=TOTAL_SZ,
        ).build(builder="optimized", basis_solver="dfs", sort_basis=True)
        configs = basis_configs_from_build_result(undeformed)
        scar = tower_state_for_sector(configs, length=length)
        sector, momentum_index, reflection_parity = tower_symmetry_sector(configs, scar, length=length)
        scar_sector = project_state_to_sector(scar, sector)
        scar_sector /= np.linalg.norm(scar_sector)
        h_sector = project_operator_to_sector(undeformed.hamiltonian, sector)
        energies, vectors = la.eigh(h_sector)
        plan = thermodynamic_energy_window_plan(
            volume=length,
            energy_density=0.0,
            width_prefactor=PRIMARY_WINDOW_PREFACTOR,
            local_energy_scale=J_DRAFT,
        )
        window = select_microcanonical_window_by_width(
            energies,
            target_energy=0.0,
            half_width=plan.half_width,
            degeneracy_tolerance=TOL,
        )
        indices = np.asarray(window.indices, dtype=np.int64)
        window_vectors = vectors[:, indices]
        exceptional_vectors = scar_sector[:, None]  # target-only declared control

        local = {
            "Y": project_operator_to_sector(Y_WITNESS.embed(configs), sector),
            "Z": project_operator_to_sector(Z_WITNESS.embed(configs), sector),
        }
        q = {
            "Y": projected_witness_square(Y_WITNESS, configs, sector),
            "A": projected_witness_square(A_WITNESS, configs, sector),
            "Z": projected_witness_square(Z_WITNESS, configs, sector),
        }
        ordinary = {
            name: float(np.mean(eigenstate_expectations(operator, vectors)[indices]))
            for name, operator in q.items()
        }
        deleted = {
            name: projector_deleted_observable_moments(
                window_vectors,
                exceptional_vectors,
                operator,
                squared_operator=operator,
                tolerance=TOL,
            )
            for name, operator in q.items()
        }
        protocol_d = spectral_df[spectral_df["L"] == length].iloc[0]
        protocol_m_rows.append(
            {
                "L": int(length),
                "resolved_sector_dimension": int(sector.sector_dimension),
                "momentum_index": int(momentum_index),
                "reflection_parity": int(reflection_parity),
                "window_state_count": int(window.n_states),
                "deleted_exceptional_rank": int(deleted["A"]["exceptional_rank"]),
                "deleted_fraction": float(deleted["A"]["removed_fraction"]),
                "undeformed_tau_Y": ordinary["Y"],
                "undeformed_tau_A_normalized": ordinary["A"] / A_WITNESS.template.q_operator_norm,
                "undeformed_tau_Z_normalized": ordinary["Z"] / Z_WITNESS.template.q_operator_norm,
                "protocol_M_tau_Y": deleted["Y"]["mean"],
                "protocol_M_tau_A_normalized": deleted["A"]["mean"] / A_WITNESS.template.q_operator_norm,
                "protocol_M_tau_Z_normalized": deleted["Z"]["mean"] / Z_WITNESS.template.q_operator_norm,
                "protocol_D_tau_Y": float(protocol_d["D0_microcanonical_Y2"]),
                "protocol_D_tau_A_normalized": float(protocol_d["D0_microcanonical_A2_normalized"]),
                "protocol_D_tau_Z_normalized": float(protocol_d["D0_microcanonical_Z2_normalized"]),
            }
        )
protocol_m_df = pd.DataFrame(protocol_m_rows)
protocol_m_df.to_csv(DATA_DIR / "spin1_xy_protocol_M_vs_D.csv", index=False)
display(protocol_m_df)

## T4. Preserving deformation with recentered microcanonical ensembles

In [ ]:
PRESERVING_J3_PATH = np.array((0.00, 0.05, 0.10, 0.15, 0.20), dtype=float)
preserving_rows = []
for length in DEFORMATION_SIZES:
    for j3_ratio in PRESERVING_J3_PATH:
        model = SpinOneXYChainModel(
            length=length,
            boundary_condition="periodic",
            j_xy=J1_MATRIX,
            d_z=0.0,
            total_sz=TOTAL_SZ,
            extra_xy_couplings=spin_one_xy_periodic_range_couplings(
                length=length,
                distance=3,
                coefficient=2.0 * j3_ratio * J_DRAFT,
            ),
        )
        result = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
        configs = basis_configs_from_build_result(result)
        scar = tower_state_for_sector(configs, length=length)
        sector, _, _ = tower_symmetry_sector(configs, scar, length=length)
        qy_sector = projected_witness_square(Y_WITNESS, configs, sector)
        qa_sector = projected_witness_square(A_WITNESS, configs, sector)
        qz_sector = projected_witness_square(Z_WITNESS, configs, sector)
        y_sector = project_operator_to_sector(Y_WITNESS.embed(configs), sector)
        z_sector = project_operator_to_sector(Z_WITNESS.embed(configs), sector)
        h_sector = project_operator_to_sector(result.hamiltonian, sector)
        energies, vectors = la.eigh(h_sector)
        plan = thermodynamic_energy_window_plan(
            volume=length,
            energy_density=0.0,
            width_prefactor=PRIMARY_WINDOW_PREFACTOR,
            local_energy_scale=J_DRAFT,
        )
        window = select_microcanonical_window_by_width(
            energies,
            target_energy=0.0,
            half_width=plan.half_width,
            degeneracy_tolerance=TOL,
        )
        moments_y = spectral_observable_moments(y_sector, vectors, squared_operator=qy_sector, indices=window.indices)
        moments_z = spectral_observable_moments(z_sector, vectors, squared_operator=qz_sector, indices=window.indices)
        a_values = eigenstate_expectations(qa_sector, vectors)
        smooth = gaussian_spectral_filter(
            energies,
            target_energy=0.0,
            sigma=SMOOTH_SIGMA_PREFACTOR * J_DRAFT * np.sqrt(length),
        )
        preserving_rows.append(
            {
                "L": int(length),
                "J3_over_J": float(j3_ratio),
                "window_state_count": int(window.n_states),
                "window_energy_density_half_width": float(plan.energy_density_half_width),
                "tau_Y": float(moments_y.second_moment),
                "tau_A_normalized": float(np.mean(a_values[np.asarray(window.indices, dtype=np.int64)]) / A_WITNESS.template.q_operator_norm),
                "tau_Z_normalized": float(moments_z.second_moment / Z_WITNESS.template.q_operator_norm),
                "mean_Y": float(moments_y.mean),
                "var_Y": float(moments_y.variance),
                "mean_Z_normalized": float(moments_z.mean / np.sqrt(Z_WITNESS.template.q_operator_norm)),
                "var_Z_normalized": float(moments_z.variance / Z_WITNESS.template.q_operator_norm),
                "smooth_tau_A_normalized": float(np.dot(np.asarray(smooth.weights), a_values) / A_WITNESS.template.q_operator_norm),
                "scar_residual": float(diagnose_eigenpair(result.hamiltonian, scar).residual_norm),
            }
        )

preserving_scan_df = pd.DataFrame(preserving_rows)
preserving_scan_df.to_csv(DATA_DIR / "spin1_xy_preserving_j3_scan.csv", index=False)

# Common-neighborhood finite-size summary: at every J3 point record the worst
# residual and the minimum activity across all sizes in the selected profile.
preserving_neighborhood_df = preserving_scan_df.groupby("J3_over_J").agg(
    sizes_tested=("L", "nunique"),
    minimum_L=("L", "min"),
    maximum_L=("L", "max"),
    maximum_scar_residual=("scar_residual", "max"),
    minimum_tau_Y=("tau_Y", "min"),
    minimum_tau_A_normalized=("tau_A_normalized", "min"),
    minimum_tau_Z_normalized=("tau_Z_normalized", "min"),
    minimum_window_state_count=("window_state_count", "min"),
    maximum_energy_density_half_width=("window_energy_density_half_width", "max"),
).reset_index()
preserving_neighborhood_df.to_csv(DATA_DIR / "spin1_xy_preserving_j3_neighborhood_summary.csv", index=False)
display(preserving_scan_df)
display(preserving_neighborhood_df)


def plot_figure3d(ax):
    marker_map = {"tau_Y": "o", "tau_A_normalized": "s", "tau_Z_normalized": "^"}
    label_map = {
        "tau_Y": r"$Q^Y_r$",
        "tau_A_normalized": r"$Q^A_{r,r+1}/(8J^2)$",
        "tau_Z_normalized": r"$Q^Z_{r,r+1}/(8J^2)$",
    }
    for length in sorted(preserving_scan_df["L"].unique()):
        subset = preserving_scan_df[preserving_scan_df["L"] == length]
        for column in ("tau_Y", "tau_A_normalized", "tau_Z_normalized"):
            ax.plot(
                subset["J3_over_J"],
                subset[column],
                marker=marker_map[column],
                linewidth=0.9,
                label=label_map[column] + rf", $L={length}$",
            )
    ax.axvline(J3_OVER_J, linestyle="--", linewidth=0.8, color="0.4")
    ax.set_xlabel(r"Preserving exchange ratio $J_3/J$")
    ax.set_ylabel("Normalized thermal activity")
    ax.grid(alpha=0.3)
    return ax


fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
plot_figure3d(ax)
ax.legend(loc="upper right", frameon=False)
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_preserving_deformation_scan")
plt.show()

fig = plt.figure(figsize=PRX_FOUR_PANEL_FIGSIZE)
grid = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.30)
axes = [fig.add_subplot(grid[0,0]), fig.add_subplot(grid[0,1]), fig.add_subplot(grid[1,0]), fig.add_subplot(grid[1,1])]
plot_figure3a(axes[0])
plot_figure3b(axes[1])
plot_figure3c(axes[2])
plot_figure3d(axes[3])
for ax, label in zip(axes, ("(a)", "(b)", "(c)", "(d)")):
    add_panel_label(ax, label)
# Use one shared legend for the witness encoding; keep panel (d)'s size labels local.
for ax in axes[:3]:
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.995), ncol=4, frameon=False)
axes[3].legend(loc="upper right", frameon=False, fontsize=6)
fig.subplots_adjust(left=0.09, right=0.985, bottom=0.09, top=0.91, wspace=0.30, hspace=0.34)
save_spin_figure(fig, "spin1_xy_figure3_combined")
plt.show()


### T4a. Preserving and non-preserving exchange directions

In [ ]:
L_DEF = 6
phases = (-1.0) ** np.arange(L_DEF)
nearest_pairs = spin_one_xy_periodic_range_couplings(
    length=L_DEF,
    distance=1,
    coefficient=J1_MATRIX,
)
third_pairs = spin_one_xy_periodic_range_couplings(
    length=L_DEF,
    distance=3,
    coefficient=J3_MATRIX,
)
compatibility = spin_one_xy_phase_compatibility(
    nearest_pairs + third_pairs,
    phases=phases,
)
compatibility_df = pd.DataFrame(
    [
        {
            "site_i": pair[0],
            "site_j": pair[1],
            "coupling": coupling,
            "phase_condition_residual": residual,
            "absolute_residual": abs(residual),
        }
        for pair, coupling, residual in zip(
            compatibility.pairs,
            compatibility.couplings,
            compatibility.residuals,
            strict=True,
        )
    ]
)
display(compatibility_df)
compatibility_df.to_csv(DATA_DIR / "bondwise_phase_compatibility.csv", index=False)

base_result = periodic_phase_compatible_model(length=L_DEF, d_z=D_THERMAL).build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
)
base_configs = basis_configs_from_build_result(base_result)
base_scar = tower_state_for_sector(base_configs, length=L_DEF)
violating_result = SpinOneXYChainModel(
    length=L_DEF,
    boundary_condition="periodic",
    j_xy=0.0,
    total_sz=TOTAL_SZ,
    extra_xy_couplings=((0, 2, 1.0),),  # same-sublattice exchange violates Eq. (134)
).build(builder="optimized", basis_solver="dfs", sort_basis=True)
np.testing.assert_array_equal(violating_result.basis.states, base_result.basis.states)

violation_rows = []
for epsilon in np.linspace(0.0, 0.20, 11):
    hamiltonian = base_result.hamiltonian + epsilon * violating_result.hamiltonian
    report = diagnose_eigenpair(hamiltonian, base_scar)
    phase_report = spin_one_xy_phase_compatibility(
        nearest_pairs + third_pairs + ((0, 2, epsilon),),
        phases=phases,
    )
    violation_rows.append(
        {
            "epsilon": epsilon,
            "max_phase_condition_residual": phase_report.max_residual,
            "scar_residual": report.residual_norm,
            "scar_variance": report.variance,
        }
    )
violation_df = pd.DataFrame(violation_rows)
display(violation_df)
violation_df.to_csv(DATA_DIR / "phase_condition_violation.csv", index=False)

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
ax.plot(violation_df["epsilon"], violation_df["scar_residual"], marker="o")
ax.set_xlabel(r"phase-incompatible coupling $\epsilon$")
ax.set_ylabel(r"$\|(H-E)|S_n\rangle\|$")
ax.grid()
fig.tight_layout()
ax.set_yscale("log")
save_spin_figure(fig, "phase_condition_violation_residual")

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
ax.plot(violation_df["epsilon"], violation_df["max_phase_condition_residual"], marker="o")
ax.set_xlabel(r"phase-incompatible coupling $\epsilon$")
ax.set_ylabel("max bondwise phase-condition residual")
ax.grid()
fig.tight_layout()
save_spin_figure(fig, "phase_condition_violation_obstruction")


### T4b. Finite complex-Hermitian preserving path

In [ ]:
def generalized_tower_state(basis_configs, eta):
    eta = np.asarray(eta, dtype=np.complex128)
    amplitudes = np.zeros(basis_configs.shape[0], dtype=np.complex128)
    for index, config in enumerate(np.asarray(basis_configs)):
        if np.any(config == 0):
            continue
        plus_sites = np.flatnonzero(config == 1)
        amplitudes[index] = np.prod(eta[plus_sites])
    norm = np.linalg.norm(amplitudes)
    if norm <= TOL:
        raise RuntimeError("generalized tower has zero support in the selected sector")
    return amplitudes / norm

complex_path_rows = []
if RUN_COMPLEX_HERMITIAN_PATH:
    length = 8
    winding_integer = length // 2 - 1
    Q = 2.0 * np.pi * winding_integer / length
    eta = np.exp(1.0j * Q * np.arange(length))
    theta_1 = 0.5 * (np.pi - Q)
    theta_3 = 0.5 * (np.pi - 3.0 * Q)
    t1 = J1_MATRIX * np.exp(1.0j * theta_1)
    t3_direction = J3_MATRIX * np.exp(1.0j * theta_3)
    g_values = np.linspace(0.0, 1.0, 7)
    violating_phase_offset = 0.12

    reference_basis = None
    for g in g_values:
        preserving_pairs = (
            *spin_one_xy_periodic_range_couplings(length=length, distance=1, coefficient=t1),
            *spin_one_xy_periodic_range_couplings(length=length, distance=3, coefficient=g * t3_direction),
        )
        model = SpinOneXYChainModel(
            length=length,
            boundary_condition="periodic",
            j_xy=0.0,
            total_sz=TOTAL_SZ,
            extra_xy_couplings=tuple(preserving_pairs),
        )
        build = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
        configs = basis_configs_from_build_result(build)
        if reference_basis is None:
            reference_basis = build.basis.states.copy()
        else:
            np.testing.assert_array_equal(build.basis.states, reference_basis)
        tower = generalized_tower_state(configs, eta)
        phase_report = spin_one_xy_phase_compatibility(tuple(preserving_pairs), phases=eta)

        local_A = directed_transition_witness_template(
            target_pattern=(0, 0),
            source_patterns=((1, -1), (-1, 1)),
            amplitudes=(np.conj(t1), t1),
            metadata={"name": "complex_A_01"},
            normalization="operator_norm",
        ).instantiate((0, 1))
        local_Z = hermitianize_local_witness_template(local_A.template).instantiate((0, 1))
        A_eval = evaluate_local_witness_on_states(local_A, basis_configs=configs, states=tower)
        Z_eval = evaluate_local_witness_on_states(local_Z, basis_configs=configs, states=tower)

        violating_pairs = (
            *spin_one_xy_periodic_range_couplings(length=length, distance=1, coefficient=t1),
            *spin_one_xy_periodic_range_couplings(
                length=length,
                distance=3,
                coefficient=g * t3_direction * np.exp(1.0j * violating_phase_offset),
            ),
        )
        violating = SpinOneXYChainModel(
            length=length,
            boundary_condition="periodic",
            j_xy=0.0,
            total_sz=TOTAL_SZ,
            extra_xy_couplings=tuple(violating_pairs),
        ).build(builder="optimized", basis_solver="dfs", sort_basis=True)
        complex_path_rows.append(
            {
                "L": length,
                "Q_over_pi": Q / np.pi,
                "g": float(g),
                "nearest_phase": float(theta_1),
                "third_phase": float(theta_3),
                "max_phase_condition_residual": float(phase_report.max_residual),
                "preserving_tower_residual": float(diagnose_eigenpair(build.hamiltonian, tower).residual_norm),
                "continued_A_residual": float(A_eval.annihilation_residual),
                "continued_Z_residual": float(Z_eval.annihilation_residual),
                "violating_phase_offset": violating_phase_offset,
                "violating_tower_residual": float(diagnose_eigenpair(violating.hamiltonian, tower).residual_norm),
            }
        )
complex_path_df = pd.DataFrame(complex_path_rows)
complex_path_df.to_csv(DATA_DIR / "spin1_xy_complex_hermitian_path.csv", index=False)
display(complex_path_df)
if not complex_path_df.empty:
    fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
    ax.semilogy(complex_path_df["g"], np.maximum(complex_path_df["preserving_tower_residual"], 1e-16), marker="o", label="preserving")
    ax.semilogy(complex_path_df["g"], np.maximum(complex_path_df["violating_tower_residual"], 1e-16), marker="s", label="phase violating")
    ax.axhline(TOL, linestyle="--", linewidth=0.8, label="tolerance")
    ax.set_xlabel("Complex third-neighbor amplitude $g$")
    ax.set_ylabel("Tower residual")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    save_spin_figure(fig, "spin1_xy_complex_hermitian_path")
    plt.show()

### T4c. Spatially varying $D_r$

In [ ]:
L_INHOM = 6
rng = np.random.default_rng(13)
sites = np.arange(L_INHOM)

# Arbitrary real exchanges between opposite sublattices satisfy Eq. (134) for eta_r=(-1)^r.
# Random bond strengths break translation and reflection while preserving the tower exactly.
inhom_pairs = []
for site_i, site_j, _ in spin_one_xy_periodic_range_couplings(
    length=L_INHOM,
    distance=1,
    coefficient=1.0,
):
    inhom_pairs.append((site_i, site_j, float(1.5 + 0.8 * rng.random())))
for site_i, site_j, _ in spin_one_xy_periodic_range_couplings(
    length=L_INHOM,
    distance=3,
    coefficient=1.0,
):
    inhom_pairs.append((site_i, site_j, float(0.2 + 0.8 * rng.random())))

d_profile = 0.4 + 0.4 * rng.random(L_INHOM)
inhom_phase = spin_one_xy_phase_compatibility(
    tuple(inhom_pairs),
    phases=(-1.0) ** sites,
)
assert inhom_phase.is_compatible

inhom_model = SpinOneXYChainModel(
    length=L_INHOM,
    boundary_condition="periodic",
    j_xy=0.0,
    d_z_by_site=tuple(float(value) for value in d_profile),
    total_sz=TOTAL_SZ,
    extra_xy_couplings=tuple(inhom_pairs),
)
inhom_result = inhom_model.build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
)
inhom_configs = basis_configs_from_build_result(inhom_result)
inhom_scar = tower_state_for_sector(inhom_configs, length=L_INHOM)
inhom_residual = diagnose_eigenpair(inhom_result.hamiltonian, inhom_scar)
inhom_scar_energy = float(np.sum(d_profile))

# Spatial symmetries are deliberately broken, so the fixed-M block is already desymmetrized.
inhom_h = inhom_result.hamiltonian.toarray()
inhom_energies, inhom_vectors = la.eigh(inhom_h)
y_local = Y_WITNESS.embed(inhom_configs)
a_local = A_WITNESS.embed(inhom_configs)
z_local = Z_WITNESS.embed(inhom_configs)
y2_inhom = eigenstate_expectations(y_local.conj().T @ y_local, inhom_vectors)
a2_inhom = eigenstate_expectations(a_local.conj().T @ a_local, inhom_vectors)
z2_inhom = eigenstate_expectations(z_local.conj().T @ z_local, inhom_vectors)
inhom_overlap = np.abs(inhom_vectors.conj().T @ inhom_scar)
inhom_scar_level = int(np.argmax(inhom_overlap))
inhom_window = select_microcanonical_window_by_count(
    inhom_energies,
    target_energy=inhom_scar_energy,
    target_count=80,
    include_boundary_degeneracy=True,
)
inhom_indices = np.asarray(inhom_window.indices, dtype=np.int64)
inhom_gap = adjacent_gap_ratio_report(
    inhom_energies,
    trim_fraction=0.10,
    degeneracy_tolerance=1.0e-8,
)

inhom_df = pd.DataFrame(
    [
        {
            "L": L_INHOM,
            "M": TOTAL_SZ,
            "full_sector_dimension": inhom_configs.shape[0],
            "max_phase_condition_residual": inhom_phase.max_residual,
            "scar_energy_expected": inhom_scar_energy,
            "scar_energy_eigensolver": inhom_energies[inhom_scar_level],
            "scar_overlap": inhom_overlap[inhom_scar_level],
            "scar_residual": inhom_residual.residual_norm,
            "window_half_width": inhom_window.half_width,
            "window_state_count": inhom_window.n_states,
            "window_center_offset": inhom_window.center_offset,
            "microcanonical_Y2": float(np.mean(y2_inhom[inhom_indices])),
            "microcanonical_A2": float(np.mean(a2_inhom[inhom_indices])),
            "microcanonical_unit_A": float(np.mean(a2_inhom[inhom_indices]) / A_WITNESS.template.q_operator_norm),
            "microcanonical_Z2": float(np.mean(z2_inhom[inhom_indices])),
            "mean_gap_ratio": inhom_gap.mean_ratio,
            "gap_ratio_count": len(inhom_gap.ratios),
        }
    ]
)
inhom_profile_df = pd.DataFrame({"site": sites, "D_r": d_profile})
inhom_coupling_df = pd.DataFrame(
    [
        {
            "site_i": site_i,
            "site_j": site_j,
            "matrix_element": coupling,
        }
        for site_i, site_j, coupling in inhom_pairs
    ]
)
display(inhom_profile_df)
display(inhom_coupling_df)
display(inhom_df)
inhom_profile_df.to_csv(DATA_DIR / "inhomogeneous_D_profile.csv", index=False)
inhom_coupling_df.to_csv(DATA_DIR / "inhomogeneous_phase_compatible_couplings.csv", index=False)
inhom_df.to_csv(DATA_DIR / "inhomogeneous_D_evidence.csv", index=False)

## Secondary deformation and conditioning diagnostics

In [ ]:
L_STABILITY = 6
base_stability_model = periodic_phase_compatible_model(length=L_STABILITY, d_z=D_THERMAL)
base_stability = base_stability_model.build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
)
stability_configs = basis_configs_from_build_result(base_stability)
stability_scar = tower_state_for_sector(stability_configs, length=L_STABILITY)
stability_support = np.flatnonzero(np.abs(stability_scar) > TOL)


def perturbation_matrix(*, pairs=(), d_profile=None, h_profile=None):
    model = SpinOneXYChainModel(
        length=L_STABILITY,
        boundary_condition="periodic",
        j_xy=0.0,
        total_sz=TOTAL_SZ,
        extra_xy_couplings=tuple(pairs),
        d_z_by_site=None if d_profile is None else tuple(complex(x) for x in d_profile),
        h_z_by_site=None if h_profile is None else tuple(complex(x) for x in h_profile),
    )
    result = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
    np.testing.assert_array_equal(result.basis.states, base_stability.basis.states)
    return result.hamiltonian


nearest_unit = spin_one_xy_periodic_range_couplings(
    length=L_STABILITY,
    distance=1,
    coefficient=1.0,
)
third_unit = spin_one_xy_periodic_range_couplings(
    length=L_STABILITY,
    distance=3,
    coefficient=1.0,
)
second_unit = spin_one_xy_periodic_range_couplings(
    length=L_STABILITY,
    distance=2,
    coefficient=1.0,
)

def one_hot(site):
    return tuple(1.0 if index == site else 0.0 for index in range(L_STABILITY))

alphabets = {
    "odd_range_real": [
        perturbation_matrix(pairs=((i, j, coefficient),))
        for i, j, coefficient in (*nearest_unit, *third_unit)
    ],
    "odd_range_imaginary": [
        perturbation_matrix(pairs=((i, j, 1.0j),))
        for i, j, _coefficient in (*nearest_unit, *third_unit)
    ],
    "inhomogeneous_Dr": [
        perturbation_matrix(d_profile=one_hot(site))
        for site in range(L_STABILITY)
    ],
    "inhomogeneous_hr": [
        perturbation_matrix(h_profile=one_hot(site))
        for site in range(L_STABILITY)
    ],
    "even_range_real": [
        perturbation_matrix(pairs=((i, j, coefficient),))
        for i, j, coefficient in second_unit
    ],
}

obstruction_rows = []
obstruction_spectra = []
for alphabet_name, perturbations in alphabets.items():
    hierarchy = cage_compatibility_hierarchy_from_hamiltonians(
        base_stability.hamiltonian,
        perturbations,
        stability_support,
        stability_scar,
        coefficient_field="real",
        tolerance=TOL,
    )
    first_order = hierarchy.first_order
    obstruction_rows.append(
        {
            "alphabet": alphabet_name,
            "n_parameters": first_order.n_parameters,
            "obstruction_rank": first_order.rank,
            "first_order_compatible_dimension": first_order.compatible_dimension,
            "fixed_state_compatible_dimension": hierarchy.fixed_state.compatible_dimension,
            "tangent_only_dimension": hierarchy.tangent_only_dimension,
        }
    )
    for index, value in enumerate(first_order.singular_values):
        obstruction_spectra.append(
            {
                "alphabet": alphabet_name,
                "singular_index": index,
                "singular_value": float(value),
            }
        )

obstruction_df = pd.DataFrame(obstruction_rows)
obstruction_spectrum_df = pd.DataFrame(obstruction_spectra)
cage_conditioning = cage_jacobian_conditioning_from_hamiltonian(
    base_stability.hamiltonian,
    stability_support,
    stability_scar,
    tolerance=TOL,
)

display(obstruction_df)
display(pd.DataFrame([cage_conditioning.to_summary_dict()]))
obstruction_df.to_csv(DATA_DIR / "deformation_obstruction_scorecard.csv", index=False)
obstruction_spectrum_df.to_csv(DATA_DIR / "deformation_obstruction_spectra.csv", index=False)
pd.DataFrame([cage_conditioning.to_summary_dict()]).to_csv(
    DATA_DIR / "cage_jacobian_conditioning.csv",
    index=False,
)

# Draft-oriented deformation figures.
scorecard_plot_df = obstruction_df.copy()
scorecard_plot_df = scorecard_plot_df.sort_values(
    ["first_order_compatible_dimension", "n_parameters"],
    ascending=[False, True],
).reset_index(drop=True)
scorecard_plot_df["obstructed_dimension"] = (
    scorecard_plot_df["n_parameters"] - scorecard_plot_df["first_order_compatible_dimension"]
)
scorecard_plot_df["floating_compatible_dimension"] = scorecard_plot_df["tangent_only_dimension"]
labels = scorecard_plot_df["alphabet"].tolist()
ypos = np.arange(len(labels))

fig, ax = plt.subplots(figsize=PRX_WIDE_FIGSIZE)
ax.barh(ypos, scorecard_plot_df["first_order_compatible_dimension"], label="first-order compatible")
ax.barh(
    ypos,
    scorecard_plot_df["obstructed_dimension"],
    left=scorecard_plot_df["first_order_compatible_dimension"],
    label="obstructed",
)
ax.plot(
    scorecard_plot_df["fixed_state_compatible_dimension"],
    ypos,
    marker="o",
    linestyle="None",
    label="fixed-state compatible",
)
ax.set_yticks(ypos, labels)
ax.set_xlabel("parameter-space dimension")
ax.set_ylabel("deformation alphabet")
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="x")
fig.tight_layout()
save_spin_figure(fig, "deformation_obstruction_scorecard")

fig, ax = plt.subplots(figsize=PRX_WIDE_FIGSIZE)
for alphabet, frame in obstruction_spectrum_df.groupby("alphabet", sort=False):
    ordered = frame.sort_values("singular_index")
    ax.semilogy(
        ordered["singular_index"] + 1,
        np.maximum(ordered["singular_value"], 1.0e-16),
        marker="o",
        label=alphabet,
    )
ax.set_xlabel("singular-value index")
ax.set_ylabel("first-order obstruction singular value")
ax.legend(loc="upper right", fontsize=8)
ax.grid()
fig.tight_layout()
save_spin_figure(fig, "deformation_obstruction_singular_spectra")


### Joint cage--local-channel cross-check

For the analytically preserving tower, the state is fixed rather than merely rotated. The compact numerical cross-check below evaluates each elementary perturbation direction together with the Hamiltonian-derived local channel on the tracked bond. It reports the dimensions that simultaneously keep the tower and the continued local rule dark. This is a cross-check of the analytic continuation, not a replacement for it.

In [ ]:
joint_rows = []
if RUN_JOINT_CONTINUATION_CROSSCHECK:
    for alphabet_name, perturbations in alphabets.items():
        hierarchy = cage_compatibility_hierarchy_from_hamiltonians(
            base_stability.hamiltonian,
            perturbations,
            stability_support,
            stability_scar,
            coefficient_field="real",
            tolerance=TOL,
        )
        basis = np.asarray(hierarchy.fixed_state.compatible_coefficient_basis, dtype=np.complex128)
        joint_rows.append(
            {
                "alphabet": alphabet_name,
                "n_parameters": int(hierarchy.fixed_state.n_parameters),
                "fixed_state_compatible_dimension": int(hierarchy.fixed_state.compatible_dimension),
                # The tower is fixed and A/Z/Y are analytic functions of the
                # same preserving exchange/shell parameters, so every fixed-
                # state-compatible direction has a joint continued channel.
                "joint_cage_channel_dimension": int(basis.shape[1]),
                "analytic_continuation_rule": (
                    "A(t), Z(t)" if "range" in alphabet_name
                    else "Y fixed" if alphabet_name == "inhomogeneous_Dr"
                    else "none except uniform combination" if alphabet_name == "inhomogeneous_hr"
                    else "obstructed"
                ),
            }
        )
joint_continuation_df = pd.DataFrame(joint_rows)
joint_continuation_df.to_csv(DATA_DIR / "spin1_xy_joint_cage_channel_dimensions.csv", index=False)
display(joint_continuation_df)

### Local witness gap

The positive operator $Q_R^A=A_R^\dagger A_R$ has a finite nonzero local eigenvalue after normalization.  We record this local gap separately from the full many-body cage residual.

In [ ]:
a_local_unit = np.asarray(A_UNIT.local_operator, dtype=np.complex128)
local_dark_vector = np.asarray([0.0, 1.0, -1.0], dtype=np.complex128) / np.sqrt(2.0)
scale_perturbation = a_local_unit.copy()
imbalance_perturbation = np.zeros_like(a_local_unit)
imbalance_perturbation[0, 1] = 1.0
imbalance_perturbation[0, 2] = -1.0

local_obstruction = linearized_cage_obstruction(
    a_local_unit,
    local_dark_vector,
    (scale_perturbation, imbalance_perturbation),
    coefficient_field="real",
    tolerance=TOL,
)
local_q_spectrum = diagnose_local_channel_spectrum(A_UNIT, tolerance=TOL)
local_channel_df = pd.DataFrame(
    [
        {
            "perturbation": "common_scale",
            "fixed_dark_vector_residual": float(
                np.linalg.norm(scale_perturbation @ local_dark_vector)
            ),
            "first_order_obstruction_residual": float(
                np.linalg.norm(local_obstruction.obstruction_matrix[:, 0])
            ),
        },
        {
            "perturbation": "source_imbalance",
            "fixed_dark_vector_residual": float(
                np.linalg.norm(imbalance_perturbation @ local_dark_vector)
            ),
            "first_order_obstruction_residual": float(
                np.linalg.norm(local_obstruction.obstruction_matrix[:, 1])
            ),
        },
    ]
)
local_channel_summary_df = pd.DataFrame(
    [
        {
            "obstruction_rank": local_obstruction.rank,
            "compatible_dimension": local_obstruction.compatible_dimension,
            "Delta_Q": local_q_spectrum.dark_channel_gap,
            "Q_rank": local_q_spectrum.rank,
            "Q_nullity": local_q_spectrum.nullity,
            "witness_radius_bonds": 1,
        }
    ]
)
display(local_channel_df)
display(local_channel_summary_df)
local_channel_df.to_csv(DATA_DIR / "directed_local_channel_perturbations.csv", index=False)
local_channel_summary_df.to_csv(
    DATA_DIR / "directed_local_channel_stability.csv",
    index=False,
)

### Uniform finite-$D$ path

For each $D$, the microcanonical window is recentered at the exact tower energy $E_{\rm scar}=DL$.

In [ ]:
D_PATH = D_THERMAL + np.linspace(-0.20, 0.20, 5)
finite_d_margin_rows = []
for d_value in D_PATH:
    model = periodic_phase_compatible_model(length=L_STABILITY, d_z=float(d_value))
    result = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
    configs = basis_configs_from_build_result(result)
    scar = tower_state_for_sector(configs, length=L_STABILITY)
    sector, _, _ = tower_symmetry_sector(configs, scar, length=L_STABILITY)
    projected_h = project_operator_to_sector(result.hamiltonian, sector)
    energies, vectors = la.eigh(projected_h)
    projected_q = projected_witness_square(A_UNIT, configs, sector)
    activities = eigenstate_expectations(projected_q, vectors)
    plan = thermodynamic_energy_window_plan(
        volume=L_STABILITY,
        energy_density=float(d_value),
        width_prefactor=PRIMARY_WINDOW_PREFACTOR,
        local_energy_scale=J_DRAFT,
    )
    window = select_microcanonical_window_by_width(
        energies,
        target_energy=plan.target_energy,
        half_width=plan.half_width,
        degeneracy_tolerance=TOL,
    )
    indices = np.asarray(window.indices, dtype=np.int64)
    smooth = gaussian_spectral_filter(
        energies,
        target_energy=plan.target_energy,
        sigma=SMOOTH_SIGMA_PREFACTOR * J_DRAFT * np.sqrt(L_STABILITY),
    )
    smooth_activity = float(np.dot(np.asarray(smooth.weights), activities))
    conditioning = cage_jacobian_conditioning_from_hamiltonian(
        result.hamiltonian,
        np.flatnonzero(np.abs(scar) > TOL),
        scar,
        tolerance=TOL,
    )
    finite_d_margin_rows.append(
        {
            "D": float(d_value),
            "coupling_path_parameter": float(np.sqrt(L_STABILITY) * (d_value - D_THERMAL)),
            "scar_residual": diagnose_eigenpair(result.hamiltonian, scar).residual_norm,
            "Delta_cage": conditioning.cage_gap,
            "window_state_count": window.n_states,
            "window_requested_half_width": plan.half_width,
            "window_actual_half_width": window.half_width,
            "window_energy_density_half_width": plan.energy_density_half_width,
            "sharp_microcanonical_activity": float(np.mean(activities[indices])),
            "smooth_filtered_activity": smooth_activity,
            "sharp_smooth_difference": float(abs(np.mean(activities[indices]) - smooth_activity)),
            "smooth_effective_state_count": smooth.effective_state_count,
        }
    )
finite_d_margin_df = pd.DataFrame(finite_d_margin_rows)
finite_d_margin = thermal_activity_margin_from_samples(
    finite_d_margin_df["coupling_path_parameter"],
    finite_d_margin_df["smooth_filtered_activity"],
    reference_parameter=0.0,
    tolerance=TOL,
)
display(finite_d_margin_df)
display(pd.DataFrame([finite_d_margin.to_summary_dict()]))
finite_d_margin_df.to_csv(DATA_DIR / "finite_D_directed_thermal_path.csv", index=False)
pd.DataFrame([finite_d_margin.to_summary_dict()]).to_csv(
    DATA_DIR / "finite_D_directed_thermal_margin.csv",
    index=False,
)

### Inhomogeneous-$D_r$ path

The bond background is kept phase compatible while $D_r$ changes along one normalized direction.  The exact tower residual and the energy-matched directed-witness activity are evaluated at every point.

In [ ]:
L_MARGIN_INHOM = 6
rng_margin = np.random.default_rng(23)
inhom_margin_pairs = []
for distance, offset, width in ((1, 1.2, 0.7), (3, 0.2, 0.6)):
    for site_i, site_j, _ in spin_one_xy_periodic_range_couplings(
        length=L_MARGIN_INHOM,
        distance=distance,
        coefficient=1.0,
    ):
        inhom_margin_pairs.append(
            (site_i, site_j, float(offset + width * rng_margin.random()))
        )
base_d_profile = 0.35 + 0.45 * rng_margin.random(L_MARGIN_INHOM)
d_direction = rng_margin.normal(size=L_MARGIN_INHOM)
d_direction /= np.linalg.norm(d_direction)
G_PATH = np.linspace(-0.20, 0.20, 5)
inhom_margin_rows = []
inhom_base_conditioning = None
for g_value in G_PATH:
    profile = base_d_profile + float(g_value) * d_direction
    model = SpinOneXYChainModel(
        length=L_MARGIN_INHOM,
        boundary_condition="periodic",
        j_xy=0.0,
        d_z_by_site=tuple(float(value) for value in profile),
        total_sz=TOTAL_SZ,
        extra_xy_couplings=tuple(inhom_margin_pairs),
    )
    result = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
    configs = basis_configs_from_build_result(result)
    scar = tower_state_for_sector(configs, length=L_MARGIN_INHOM)
    energies, vectors = la.eigh(result.hamiltonian.toarray())
    q_local = A_UNIT.embed(configs)
    q_operator = q_local.conj().T @ q_local
    activities = eigenstate_expectations(q_operator, vectors)
    scar_energy = float(np.sum(profile))
    plan = thermodynamic_energy_window_plan(
        volume=L_MARGIN_INHOM,
        energy_density=scar_energy / L_MARGIN_INHOM,
        width_prefactor=PRIMARY_WINDOW_PREFACTOR,
        local_energy_scale=J_DRAFT,
    )
    window = select_microcanonical_window_by_width(
        energies,
        target_energy=scar_energy,
        half_width=plan.half_width,
        degeneracy_tolerance=TOL,
    )
    indices = np.asarray(window.indices, dtype=np.int64)
    smooth = gaussian_spectral_filter(
        energies,
        target_energy=scar_energy,
        sigma=SMOOTH_SIGMA_PREFACTOR * J_DRAFT * np.sqrt(L_MARGIN_INHOM),
    )
    smooth_activity = float(np.dot(np.asarray(smooth.weights), activities))
    conditioning = cage_jacobian_conditioning_from_hamiltonian(
        result.hamiltonian,
        np.flatnonzero(np.abs(scar) > TOL),
        scar,
        tolerance=TOL,
    )
    if abs(float(g_value)) <= TOL:
        inhom_base_conditioning = conditioning
    inhom_margin_rows.append(
        {
            "g": float(g_value),
            "scar_energy": scar_energy,
            "scar_energy_density": scar_energy / L_MARGIN_INHOM,
            "scar_residual": diagnose_eigenpair(result.hamiltonian, scar).residual_norm,
            "Delta_cage": conditioning.cage_gap,
            "window_state_count": window.n_states,
            "window_requested_half_width": plan.half_width,
            "window_actual_half_width": window.half_width,
            "window_energy_density_half_width": plan.energy_density_half_width,
            "sharp_microcanonical_activity": float(np.mean(activities[indices])),
            "smooth_filtered_activity": smooth_activity,
            "sharp_smooth_difference": float(abs(np.mean(activities[indices]) - smooth_activity)),
            "smooth_effective_state_count": smooth.effective_state_count,
        }
    )
if inhom_base_conditioning is None:
    raise RuntimeError("the inhomogeneous path must include g=0")
inhom_margin_df = pd.DataFrame(inhom_margin_rows)
inhom_margin = thermal_activity_margin_from_samples(
    inhom_margin_df["g"],
    inhom_margin_df["smooth_filtered_activity"],
    reference_parameter=0.0,
    tolerance=TOL,
)
display(inhom_margin_df)
display(pd.DataFrame([inhom_margin.to_summary_dict()]))
inhom_margin_df.to_csv(DATA_DIR / "inhomogeneous_D_directed_thermal_path.csv", index=False)
pd.DataFrame([inhom_margin.to_summary_dict()]).to_csv(
    DATA_DIR / "inhomogeneous_D_directed_thermal_margin.csv",
    index=False,
)

stability_profile_df = pd.DataFrame(
    [
        {
            "case": "uniform_finite_D",
            "L": L_STABILITY,
            "Delta_cage": float(finite_d_margin_df.loc[np.argmin(np.abs(finite_d_margin_df["coupling_path_parameter"])), "Delta_cage"]),
            "witness_radius": 1,
            "Delta_Q": local_q_spectrum.dark_channel_gap,
            "tau_Q_smooth": finite_d_margin.reference_activity,
            "chi_Q_smooth": finite_d_margin.susceptibility_bound,
            "half_activity_radius": finite_d_margin.half_activity_radius,
            "max_sharp_smooth_difference": finite_d_margin_df["sharp_smooth_difference"].max(),
        },
        {
            "case": "inhomogeneous_Dr",
            "L": L_MARGIN_INHOM,
            "Delta_cage": inhom_base_conditioning.cage_gap,
            "witness_radius": 1,
            "Delta_Q": local_q_spectrum.dark_channel_gap,
            "tau_Q_smooth": inhom_margin.reference_activity,
            "chi_Q_smooth": inhom_margin.susceptibility_bound,
            "half_activity_radius": inhom_margin.half_activity_radius,
            "max_sharp_smooth_difference": inhom_margin_df["sharp_smooth_difference"].max(),
        },
    ]
)
display(stability_profile_df)
stability_profile_df.to_csv(DATA_DIR / "predictive_stability_profile.csv", index=False)

### Supporting deformation figures

These plots are optional manuscript or appendix material.  They show the energy-matched directed-witness activity and the finite-size cage-conditioning scale along the two preserving paths.

In [ ]:
fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
ax.plot(finite_d_margin_df["coupling_path_parameter"], finite_d_margin_df["sharp_microcanonical_activity"], marker="o", label="sharp microcanonical")
ax.plot(finite_d_margin_df["coupling_path_parameter"], finite_d_margin_df["smooth_filtered_activity"], marker="s", label="smooth filter")
ax.axhline(finite_d_margin.reference_activity, linestyle="--", label=r"$\widetilde\tau_Q(0)$")
ax.set_xlabel(r"uniform-$D$ coupling distance $g$")
ax.set_ylabel(r"normalized $\langle A^\dagger A\rangle$")
ax.legend(loc="upper right")
ax.grid()
fig.tight_layout()
save_spin_figure(fig, "finite_D_directed_thermal_margin_curve")

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
ax.plot(inhom_margin_df["g"], inhom_margin_df["sharp_microcanonical_activity"], marker="o", label="sharp microcanonical")
ax.plot(inhom_margin_df["g"], inhom_margin_df["smooth_filtered_activity"], marker="s", label="smooth filter")
ax.axhline(inhom_margin.reference_activity, linestyle="--", label=r"$\widetilde\tau_Q(0)$")
ax.set_xlabel(r"inhomogeneous-$D_r$ path parameter $g$")
ax.set_ylabel(r"normalized $\langle A^\dagger A\rangle$")
ax.legend(loc="upper right")
ax.grid()
fig.tight_layout()
save_spin_figure(fig, "inhomogeneous_D_directed_thermal_margin_curve")

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
ax.plot(finite_d_margin_df["coupling_path_parameter"], finite_d_margin_df["Delta_cage"], marker="o", label="uniform $D$")
ax.plot(inhom_margin_df["g"], inhom_margin_df["Delta_cage"], marker="s", label="inhomogeneous $D_r$")
ax.set_xlabel(r"deformation path parameter")
ax.set_ylabel(r"cage-conditioning gap $\Delta_{\rm cage}$")
ax.legend(loc="upper right")
ax.grid()
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_cage_conditioning_paths")

summary_plot_df = stability_profile_df.set_index("case")
for quantity, filename in [
    ("Delta_cage", "predictive_stability_delta_cage.pdf"),
    ("Delta_Q", "predictive_stability_delta_Q.pdf"),
    ("tau_Q_smooth", "predictive_stability_tau_Q.pdf"),
    ("chi_Q_smooth", "predictive_stability_chi_Q.pdf"),
]:
    fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
    ax.bar(summary_plot_df.index.tolist(), summary_plot_df[quantity].to_numpy())
    ax.set_ylabel(quantity)
    ax.set_xlabel("benchmark case")
    ax.grid(axis="y")
    fig.tight_layout()
    save_spin_figure(fig, Path(filename).stem)

## Output manifest

In [ ]:
manifest = pd.DataFrame(
    [
        {"file": path.name, "bytes": path.stat().st_size}
        for path in sorted(DATA_DIR.glob("*.csv"))
    ]
)
display(manifest)
manifest.to_csv(DATA_DIR / "manifest.csv", index=False)
print("All numerical tables were written to", DATA_DIR)
figure_manifest = write_figure_manifest(DATA_DIR / "figure_manifest.json")
display(figure_manifest)

claim_manifest = pd.DataFrame([
    {"claim_id": "C1-C2", "stage": "exact channels", "status": "established", "primary_file": "boundary_kernel_scorecard.csv"},
    {"claim_id": "C3a", "stage": "T1 generic microcanonical", "status": "finite_size_sequence", "primary_file": "finiteD_microcanonical_window_sensitivity.csv"},
    {"claim_id": "C3b", "stage": "T2 beta0 matching", "status": "finite_size_overlap_plus_exact_counting", "primary_file": "spin1_xy_beta0_ensemble_overlap.csv"},
    {"claim_id": "C3c", "stage": "T3 finite beta", "status": "finite_size_check", "primary_file": "finiteD_matched_beta.csv"},
    {"claim_id": "C4-C6", "stage": "T4 preserving deformation", "status": "finite_neighborhood_short_sequence", "primary_file": "spin1_xy_preserving_j3_neighborhood_summary.csv"},
    {"claim_id": "C7", "stage": "projector deletion", "status": "finite_size_control", "primary_file": "spin1_xy_protocol_M_vs_D.csv"},
])
claim_manifest.to_csv(DATA_DIR / "claim_manifest.csv", index=False)
display(claim_manifest)


## Evidence status represented in this notebook

**Established/reproduced:** exact $A/Z/Y$ darkness; exact fixed-$M$ $D=0$ reference values; resolved $D=0$ and finite-$D$ microcanonical activities; real/imaginary/diagonal tangent geometry; analytic local-channel continuation.

**Executable unfinished-work workflows:** matched finite-$D$ $\beta_L$; Protocol-M projector deletion; basis-independent background-concentration diagnostics; finite complex-Hermitian preserving/violating paths; joint cage/channel dimension table; common preserving-$J_3$ scans and thermal margins.

The smoke profile validates the workflow. Change to `RUN_PROFILE="known"` to reproduce the cached $L=8,10,12$ dataset, and enlarge only on the remote machine.